# Notebook 03: Baseline Convolutional Neural Network

**Course:** DS 4889 Machine Learning, United International University

**Course outcomes:** CO4 (feature construction and model building), CO5 (sound validation and interpretation)

**Lecture mapping:** Lecture 08 (neural network fundamentals, backpropagation, regularisation), Lecture 09 (optimisers and training dynamics), Lecture 11 (convolutional neural networks)

## Objective

This notebook establishes the reference performance of the project. The research question asks whether
unsupervised Gaussian Mixture Model pre-segmentation improves convolutional classification of brain
magnetic resonance images. A question phrased in this comparative form cannot be answered without a
control condition, so the purpose here is to train and evaluate a convolutional network that receives
only the deterministically preprocessed images, with no segmentation stage whatsoever. Every number
reported in notebook 04 and notebook 05 is interpreted relative to the values produced below.

## Why the experiment is constructed in this way

A performance difference between two pipelines is attributable to the component under study only if
every other component is held constant. For that reason the architecture defined in `src.cnn_model`,
the preprocessing chain defined in `src.preprocessing`, the random seed stored in
`config.RANDOM_SEED`, the stratified split produced by `src.data_loader` and the augmentation policy
are all reused without modification in notebook 04. The single factor that varies between the two
experiments is the presence or absence of the Gaussian Mixture attention transformation applied to the
input tensor. This design turns an informal comparison into a controlled one, and it is what permits
the paired statistical test carried out in notebook 05.


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

sys.path.append('..')
from src import config, data_loader, preprocessing, cnn_model, evaluate

# Seeding Python, NumPy and TensorFlow before any tensor is created is what makes the weight
# initialisation, the shuffling order and the augmentation stream reproducible across this
# notebook and notebook 04.
cnn_model.set_global_seed()

sns.set_theme(style='whitegrid', context='notebook')

print('TensorFlow version:', tf.__version__)

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print('GPU devices visible:', [device.name for device in gpu_devices])
else:
    print('No GPU is visible; training will proceed on the CPU and will take longer.')

print('Random seed:', config.RANDOM_SEED)
print('Input shape:', config.IMAGE_SHAPE)
print('Classes:', config.CLASS_LABELS)


## Data loading and splitting

The Kaggle distribution of the dataset provides two folders, `Training` and `Testing`. Training a model
on the first and reporting results on the second would be methodologically acceptable only if no
decision whatsoever were informed by the second folder. In practice the number of epochs, the moment
at which early stopping fires and the selection of the checkpoint all require a performance signal
during training. If that signal came from the test folder, the reported test score would be optimistic
because the test data would have influenced model selection.

A three way split resolves this. The `Training` folder is partitioned into a training subset and a
validation subset using a stratified split, which preserves the class proportions in both parts and
therefore keeps the validation estimate unbiased with respect to the mild class imbalance present in
the data. The training subset supplies the gradients, the validation subset supplies the model
selection signal, and the `Testing` folder is held out entirely. The test split is read once, at the
end of the notebook, after all architectural and training decisions have been frozen. Treating it in
this way means the final numbers can be interpreted as an estimate of generalisation to unseen scans
rather than as a measure of how carefully the pipeline was tuned.


In [ ]:
# Set LIMIT_PER_CLASS to a small integer, for example 40, to run the entire notebook as a fast
# smoke test before committing to a full training run. Use None for the real experiment.
LIMIT_PER_CLASS = None

load_start = time.time()

x_train_raw, y_train_full, train_paths = data_loader.load_split(
    config.TRAIN_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=LIMIT_PER_CLASS,
    show_progress=True,
)

x_test_raw, y_test, test_paths = data_loader.load_split(
    config.TEST_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=LIMIT_PER_CLASS,
    show_progress=True,
)

print('Loading completed in {0:.1f} seconds.'.format(time.time() - load_start))
print('Raw training images:', x_train_raw.shape, x_train_raw.dtype)
print('Raw training labels:', y_train_full.shape, y_train_full.dtype)
print('Raw testing images: ', x_test_raw.shape, x_test_raw.dtype)
print('Raw testing labels: ', y_test.shape, y_test.dtype)


In [ ]:
# The preprocessing chain is deterministic: resize, CLAHE contrast enhancement and normalisation
# to the unit interval. Applying the identical function to every split guarantees that the three
# tensors are produced by exactly the same operations.
x_train_full = preprocessing.preprocess_batch(x_train_raw, image_size=config.IMAGE_SIZE)
x_test = preprocessing.preprocess_batch(x_test_raw, image_size=config.IMAGE_SIZE)

x_train, x_val, y_train, y_val = data_loader.stratified_validation_split(x_train_full, y_train_full)

print('Training subset:  ', x_train.shape)
print('Validation subset:', x_val.shape)
print('Test set:         ', x_test.shape)
print('Intensity range after preprocessing: [{0:.3f}, {1:.3f}]'.format(
    float(x_train.min()), float(x_train.max())
))

for split_name, split_labels in (('Training', y_train), ('Validation', y_val), ('Test', y_test)):
    print()
    print('{0} class distribution'.format(split_name))
    display(data_loader.class_distribution(split_labels))


In [ ]:
# Categorical cross entropy expects a one hot target vector rather than an integer index.
y_train_cat = to_categorical(y_train, num_classes=config.NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=config.NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=config.NUM_CLASSES)

# Inverse frequency weights prevent the optimiser from reducing the loss by favouring the
# majority classes. They are normalised to a mean of one so that the effective step size
# remains comparable with unweighted training.
class_weights = data_loader.compute_class_weights(y_train)

print('Class weights applied during training')
for class_index, weight in class_weights.items():
    print('  index {0}  {1:<12} weight {2:.4f}'.format(
        class_index, config.CLASS_LABELS[class_index], weight
    ))

print()
print('One hot target shapes:', y_train_cat.shape, y_val_cat.shape, y_test_cat.shape)


## Data augmentation

The training subset contains a few thousand images, which is modest relative to the capacity of a
convolutional network. Without augmentation the network memorises individual scans and the gap between
training and validation accuracy widens early. Augmentation addresses this by presenting a different
perturbation of each image on every epoch, which enlarges the effective support of the training
distribution and therefore acts as a regulariser.

The choice of transformations is constrained by what is physically plausible in a radiology department
rather than by what is available in the software library. Rotations of up to twenty degrees,
translations of up to ten percent of the image width and height, and zoom variation of up to twenty
percent all correspond to realistic differences in patient positioning, head tilt and field of view
between acquisitions. Horizontal flipping is admissible because the brain is approximately symmetric
about the midsagittal plane, so a mirrored slice remains an anatomically coherent image.

Vertical flipping is deliberately excluded. Brain anatomy has a fixed superior to inferior
orientation: the cerebral cortex lies above, the brainstem and skull base below. A vertically flipped
slice is not an image that any scanner could produce, and training on such images would force the
network to allocate capacity to a configuration it will never encounter at inference time. Intensity
inversion and large shear are excluded for the same reason, since tissue contrast in magnetic
resonance imaging carries diagnostic meaning and must not be altered arbitrarily.


In [ ]:
train_generator = preprocessing.make_training_generator(
    x_train, y_train_cat, batch_size=config.BATCH_SIZE
)

# Inspecting one batch visually is a cheap guard against a misconfigured augmenter that would
# otherwise silently corrupt every training image.
augmented_images, augmented_labels = next(train_generator)

figure, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for axis, image, label in zip(axes.ravel(), augmented_images[:8], augmented_labels[:8]):
    axis.imshow(np.clip(image, 0.0, 1.0))
    axis.set_title(config.CLASS_LABELS[int(np.argmax(label))], fontsize=11)
    axis.axis('off')

figure.suptitle('Representative augmented training batch', fontsize=14)
figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'augmentation_examples.png')
plt.show()

print('Batch tensor shape:', augmented_images.shape)
print('Augmented intensity range: [{0:.3f}, {1:.3f}]'.format(
    float(augmented_images.min()), float(augmented_images.max())
))


## Network architecture

The classifier is a custom convolutional network built from four sequential blocks whose filter counts
double from thirty two to two hundred and fifty six. Doubling the width while halving the spatial
resolution at each stage keeps the computational cost per block approximately constant while allowing
the representation to become progressively more abstract: early blocks respond to edges and intensity
gradients, later blocks respond to tissue level configurations that distinguish one lesion type from
another.

Each block applies convolution, then batch normalisation, then the rectified linear activation, then
max pooling, then dropout. The ordering is deliberate on two counts.

Batch normalisation precedes the activation because its purpose is to control the distribution of the
pre-activation values. Standardising the input to the non linearity keeps most units in the regime
where the rectifier has a non zero gradient, which reduces the number of units that become permanently
inactive and permits a larger learning rate without divergence. Placing normalisation after the
activation would standardise a distribution that has already been truncated at zero, which does not
serve the same purpose.

Dropout follows pooling rather than preceding it. Pooling already discards spatial detail by selecting
the maximum within each window, so applying dropout first would remove candidates from that window and
make the pooled value depend on the dropout mask in an unhelpful way. Applying dropout to the pooled
feature map instead removes whole activations from the reduced representation, which discourages
feature maps from co-adapting and forces redundant encodings of the same evidence.

The classifier head replaces the conventional flatten operation with global average pooling. Flattening
a fourteen by fourteen by two hundred and fifty six tensor into a dense layer of five hundred and twelve
units would introduce roughly twenty five million parameters, far more than a dataset of this size can
constrain, and the resulting model would overfit severely. Global average pooling reduces each feature
map to its mean, producing a two hundred and fifty six dimensional vector and cutting the parameter
count by two orders of magnitude. It also acts as a structural regulariser because it enforces a direct
correspondence between feature maps and evidence for a class, and it makes the representation invariant
to the absolute position of a lesion within the slice, which is appropriate given that a tumour may
appear anywhere in the field of view.


In [ ]:
model = cnn_model.build_cnn()
model = cnn_model.compile_model(model, learning_rate=config.LEARNING_RATE)

model.summary()

architecture_table = cnn_model.summarise_architecture(model)
display(architecture_table)

baseline_parameter_count = int(model.count_params())
print('Total parameters: {0:,}'.format(baseline_parameter_count))
print('Number of layers:', len(model.layers))


## Training protocol

The protocol is fixed before training begins and is reused without alteration in notebook 04.

**Optimiser.** Adam is used because it maintains a per parameter adaptive step size derived from
running estimates of the first and second gradient moments, which converges substantially faster than
plain stochastic gradient descent on a network that combines batch normalised convolutional layers
with a dense head.

**Initial learning rate.** A value of one times ten to the power of minus three is the standard
starting point for Adam on this class of network: large enough to traverse the initial loss surface
quickly and small enough to remain stable given batch normalisation.

**Learning rate schedule.** The `ReduceLROnPlateau` callback halves the learning rate after five
epochs without an improvement in validation accuracy, down to a floor of one times ten to the power of
minus six. A plateau indicates that the current step size is too large to enter the minimum being
approached, so reducing it permits refinement instead of oscillation.

**Loss.** Categorical cross entropy is the maximum likelihood objective for a multinomial output with
a softmax activation, so minimising it is equivalent to maximising the likelihood of the observed
labels under the model.

**Class weighting.** The inverse frequency weights computed above enter the loss so that an error on
an under represented class contributes proportionally more, which prevents the optimiser from reaching
a low loss by neglecting the smaller classes.

**Early stopping.** Training halts after ten epochs without an improvement in validation accuracy and
the best weights are restored. This prevents the network from continuing past the point of optimal
generalisation and removes the need to guess the correct epoch count in advance.

**Checkpointing.** The `ModelCheckpoint` callback writes the weights whenever validation accuracy
improves, so the artefact on disk corresponds to the best validated state rather than to the final
epoch. This matters because the final epoch may follow several degraded ones.

**Batch size.** Thirty two balances gradient noise against memory consumption: small enough to retain
the mild regularising effect of stochastic gradients, large enough for the batch normalisation
statistics to be stable.


In [ ]:
EPOCHS = config.EPOCHS
steps_per_epoch = max(1, len(x_train) // config.BATCH_SIZE)
checkpoint_path = config.MODELS_DIR / config.BASELINE_MODEL_NAME

callbacks = cnn_model.build_callbacks(checkpoint_path, monitor='val_accuracy')

print('Maximum epochs:  ', EPOCHS)
print('Steps per epoch: ', steps_per_epoch)
print('Checkpoint path: ', checkpoint_path)

training_start = time.time()

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=(x_val, y_val_cat),
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

baseline_training_seconds = time.time() - training_start

print()
print('Training wall clock time: {0:.1f} seconds ({1:.1f} minutes).'.format(
    baseline_training_seconds, baseline_training_seconds / 60.0
))


In [ ]:
evaluate.plot_training_history(
    history,
    title_prefix='Baseline CNN ',
    save_path=config.FIGURES_DIR / 'baseline_training_history.png',
)
plt.show()

epochs_completed = len(history.history['loss'])
best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
best_val_accuracy = float(np.max(history.history['val_accuracy']))
final_train_accuracy = float(history.history['accuracy'][-1])

print('Epochs completed before stopping: {0} of a maximum of {1}'.format(epochs_completed, EPOCHS))
print('Best validation accuracy: {0:.4f} at epoch {1}'.format(best_val_accuracy, best_epoch))
print('Final training accuracy:  {0:.4f}'.format(final_train_accuracy))
print('Gap between final training and best validation accuracy: {0:.4f}'.format(
    final_train_accuracy - best_val_accuracy
))


## Evaluation on the held out test set

The test split has not influenced any decision taken so far: it was not used to compute gradients, to
trigger early stopping, to select a checkpoint or to choose a hyperparameter. It is now read exactly
once, which is what allows the scores below to be interpreted as an estimate of performance on unseen
scans rather than as a tuning statistic.

Several complementary metrics are reported instead of accuracy alone. Accuracy is misleading under
class imbalance because a model can reach a respectable value while systematically failing on a
minority class. Weighted precision, recall and F1 summarise performance while accounting for class
support; macro averages give every class equal influence and therefore expose failures on the smaller
classes; Cohen's kappa corrects the observed agreement for the agreement expected by chance; and the
Matthews correlation coefficient remains informative when the error is concentrated in a single class.


In [ ]:
probabilities = model.predict(x_test, batch_size=config.BATCH_SIZE, verbose=0)
predictions = np.argmax(probabilities, axis=1)

metrics = evaluate.compute_metrics(y_test, predictions, 'CNN baseline')

metrics_series = pd.Series(
    {key: value for key, value in metrics.items() if key != 'model'},
    name='CNN baseline',
)
print('Headline metrics on the held out test set')
display(metrics_series.to_frame().round(4))

per_class = evaluate.per_class_metrics(y_test, predictions)
print('Per class metrics')
display(per_class)


In [ ]:
# The raw matrix shows absolute error counts, which is what matters when judging clinical impact.
# The row normalised matrix shows per class recall, which is what matters when comparing classes
# of unequal size.
figure, axes = plt.subplots(1, 2, figsize=(15, 5.8))

evaluate.plot_confusion_matrix(
    y_test,
    predictions,
    class_labels=config.CLASS_LABELS,
    title='Baseline CNN: counts',
    normalise=False,
    ax=axes[0],
)

evaluate.plot_confusion_matrix(
    y_test,
    predictions,
    class_labels=config.CLASS_LABELS,
    title='Baseline CNN: row normalised',
    normalise=True,
    ax=axes[1],
)

figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'baseline_confusion_matrices.png')
plt.show()


In [ ]:
# Receiver operating characteristic curves are computed one class versus the rest. They describe
# the quality of the ranking produced by the softmax output independently of the implicit decision
# threshold imposed by taking the argmax, which is useful because a screening deployment might
# prefer a different operating point.
figure, ax = plt.subplots(figsize=(7, 6))

ax, baseline_auc_scores = evaluate.plot_roc_curves(
    y_test,
    probabilities,
    class_labels=config.CLASS_LABELS,
    title='Baseline CNN: one versus rest ROC',
    ax=ax,
)

figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'baseline_roc_curves.png')
plt.show()

print('Area under the curve by class')
for label, score in baseline_auc_scores.items():
    print('  {0:<16} {1:.4f}'.format(label, score))


## Clinically weighted error analysis

Overall accuracy treats every mistake as equally costly, which does not hold in a diagnostic setting.
The two error types have markedly different consequences.

A false negative occurs when a scan that contains a tumour is assigned to the `No Tumor` class, which
is index two in `config.CLASS_LABELS`. The patient is reassured, no referral is made, and the lesion
continues to develop until symptoms force a second examination. The cost of this error is measured in
delayed treatment and in the loss of prognosis that accompanies the delay.

A false positive occurs when a healthy scan is flagged as abnormal. The consequence is an unnecessary
follow up examination, avoidable anxiety and the expenditure of radiology capacity. This is
undesirable but recoverable, because the subsequent examination corrects the error.

The asymmetry means that a model with slightly lower overall accuracy but a markedly lower false
negative rate may be the preferable system. The summary below therefore reports the two rates
separately rather than collapsing them into a single figure. A bootstrap confidence interval
accompanies the point estimates because any score computed on a finite test set carries sampling
uncertainty, and differences smaller than that uncertainty must not be reported as real effects.


In [ ]:
clinical = evaluate.clinical_error_summary(y_test, predictions, healthy_index=2)

print('Clinically weighted error summary')
for key, value in clinical.items():
    print('  {0:<24} {1}'.format(key, value))

print()
accuracy_interval = evaluate.bootstrap_confidence_interval(
    y_test, predictions, metric='accuracy', n_iterations=2000, confidence=0.95
)
f1_interval = evaluate.bootstrap_confidence_interval(
    y_test, predictions, metric='f1_weighted', n_iterations=2000, confidence=0.95
)

for interval in (accuracy_interval, f1_interval):
    print('{0}: {1:.4f} with a {2:.0%} interval of [{3:.4f}, {4:.4f}]'.format(
        interval['metric'],
        interval['point_estimate'],
        interval['confidence'],
        interval['lower_bound'],
        interval['upper_bound'],
    ))


In [ ]:
# Inspecting the failures directly is the fastest way to distinguish a model limitation from a data
# limitation: if the misclassified slices are ambiguous even to visual inspection, the ceiling is
# set by the data rather than by the architecture.
misclassified_indices = np.flatnonzero(predictions != y_test)
print('Misclassified test images: {0} of {1}'.format(len(misclassified_indices), len(y_test)))

if len(misclassified_indices) == 0:
    print('No misclassified samples are available to display.')
else:
    rng = np.random.default_rng(config.RANDOM_SEED)
    selection_size = min(12, len(misclassified_indices))
    selected = rng.choice(misclassified_indices, size=selection_size, replace=False)

    figure, axes = plt.subplots(3, 4, figsize=(15, 12))
    for axis in axes.ravel():
        axis.axis('off')

    for axis, index in zip(axes.ravel(), selected):
        axis.imshow(np.clip(x_test[index], 0.0, 1.0))
        axis.set_title(
            'True: {0}\nPredicted: {1}\nConfidence: {2:.2f}'.format(
                config.CLASS_LABELS[int(y_test[index])],
                config.CLASS_LABELS[int(predictions[index])],
                float(probabilities[index].max()),
            ),
            fontsize=10,
        )

    figure.suptitle('Misclassified test images, baseline CNN', fontsize=14)
    figure.tight_layout()
    figure.savefig(config.FIGURES_DIR / 'baseline_misclassified.png')
    plt.show()

    error_pairs = pd.DataFrame({
        'true': [config.CLASS_LABELS[int(label)] for label in y_test[misclassified_indices]],
        'predicted': [config.CLASS_LABELS[int(label)] for label in predictions[misclassified_indices]],
    })
    print('Most frequent confusion pairs')
    display(error_pairs.value_counts().head(6).to_frame('count'))


In [ ]:
# Persisting the metrics, the learning curves and the weights makes notebook 05 reproducible without
# retraining, and it fixes the exact values quoted in the written report.
metrics_path = evaluate.save_metrics(metrics, 'baseline_metrics.json')

history_payload = {
    'model': 'CNN baseline',
    'epochs_completed': int(epochs_completed),
    'best_epoch': int(best_epoch),
    'best_val_accuracy': float(best_val_accuracy),
    'training_seconds': float(baseline_training_seconds),
    'parameters': int(baseline_parameter_count),
    'batch_size': int(config.BATCH_SIZE),
    'steps_per_epoch': int(steps_per_epoch),
    'auc_scores': {key: float(value) for key, value in baseline_auc_scores.items()},
    'clinical_error_summary': {key: float(value) for key, value in clinical.items()},
    'history': {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    },
}
history_path = evaluate.save_metrics(history_payload, 'baseline_history.json')

model_path = config.MODELS_DIR / config.BASELINE_MODEL_NAME
model.save(model_path)

print('Metrics written to:', metrics_path)
print('History written to:', history_path)
print('Model written to:  ', model_path)


## Summary

The baseline convolutional network has been trained on the deterministically preprocessed images with
no segmentation stage, and evaluated once on the held out test split. The accuracy and weighted F1
printed by the metrics cell above, and stored in `results/metrics/baseline_metrics.json`, constitute
the reference point for the remainder of the project. Notebook 04 trains an architecturally identical
network on segmentation weighted inputs, and notebook 05 tests whether the difference between the two
exceeds sampling noise. Any claim about the value of Gaussian Mixture pre-segmentation is ultimately a
claim about the gap between these two numbers, which is why they are recorded explicitly rather than
described qualitatively.

The confusion matrices and the confusion pair table identify where the residual error is concentrated.
On this dataset the dominant off diagonal mass is normally the glioma and meningioma pair in both
directions, which is expected on clinical grounds: both present as space occupying masses with
overlapping intensity profiles on a single two dimensional slice, and the feature that separates them
most reliably, namely whether the lesion arises within the brain parenchyma or from the meninges at
its surface, is a location cue that is only partially visible in one slice. The second largest source
of error is typically meningioma predicted as `No Tumor`, which is the clinically costly direction and
which arises when a small lesion at the brain margin occupies few pixels after resizing to two hundred
and twenty four pixels. Pituitary cases are usually separated most cleanly because the lesion occupies
a stereotyped position at the skull base.

This error structure motivates the hypothesis examined in notebook 04. If the dominant confusions are
driven by the network failing to isolate the lesion and its position from the surrounding anatomy,
then supplying an explicit spatial prior should reduce precisely those confusions. The clinically
weighted summary and the bootstrap intervals reported above provide the two further reference
quantities against which that claim is assessed.
